# Training tables EDA (univariate + fingerprint)

Inspect the outputs from `pipelines.monthly.features` (`pipelines/monthly/features.py`):

- `data/processed/training_univariate.parquet` — one row per eligible `(dimension, level_id)` anchor month
- `data/processed/training_fingerprint.parquet` — one row per eligible 5-D fingerprint anchor month
- `data/processed/training_run.json` — feature contract, split defaults, sample-weight formula

**If files are missing**, run from the package root that contains `pipelines/`: `python -m pipelines.monthly.features` (after merged cubes exist).

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

_here = Path.cwd().resolve()
_root = None
for candidate in [_here, *_here.parents]:
    if (candidate / "pipelines" / "paths.py").is_file():
        _root = candidate
        break
if _root is None:
    raise RuntimeError(
        "Cannot find pipelines/paths.py — open this notebook with cwd inside the repo tree "
        "(same folder level as pipelines/)."
    )
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from pipelines.paths import (
    MODEL_TRAINING_RUN_JSON,
    TRAINING_FINGERPRINT_PARQUET,
    TRAINING_RUN_JSON,
    TRAINING_UNIVARIATE_PARQUET,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

TARGET_COLS = [f"y_h{h}" for h in range(1, 7)]
LAG_COLS = ["share_lag1", "share_lag2", "share_lag3"]
FEATURE_CORE = ["month_of_year", "share_t", *LAG_COLS]

print("project_root:", _root)
print("training_run:", TRAINING_RUN_JSON)
print("univ parquet:", TRAINING_UNIVARIATE_PARQUET)
print("fp parquet:  ", TRAINING_FINGERPRINT_PARQUET)

project_root: /Users/jackcdawson/Desktop/trndly/trndly
training_run: /Users/jackcdawson/Desktop/trndly/trndly/data/processed/training_run.json
univ parquet: /Users/jackcdawson/Desktop/trndly/trndly/data/processed/training_univariate.parquet
fp parquet:   /Users/jackcdawson/Desktop/trndly/trndly/data/processed/training_fingerprint.parquet


In [2]:
contract_path = Path(TRAINING_RUN_JSON)
if not contract_path.is_file():
    raise FileNotFoundError(
        f"Missing {contract_path}. Run: python -m pipelines.monthly.features"
    )

with open(contract_path) as f:
    contract = json.load(f)

uni_feats = contract["univariate_feature_cols"]
fp_feats = contract["fingerprint_feature_cols"]
print("generated_at_utc:", contract.get("generated_at_utc"))
print("calendar_strict:", json.dumps(contract.get("calendar_strict"), indent=2))
print("split_defaults:", json.dumps(contract.get("split_defaults"), indent=2))
print("sample_weight:", json.dumps(contract.get("sample_weight"), indent=2))
print("univariate features:", uni_feats)
print("fingerprint features:", fp_feats)
assert uni_feats == fp_feats == FEATURE_CORE

for name in ("univariate", "fingerprint"):
    st = contract.get("part_a_stats" if name == "univariate" else "part_b_stats", {})
    out = contract["outputs"][f"{name}_training"]
    print(f"\n{name}: rows_written={out['rows']:,} cols={len(out['cols'])} stats={st}")

generated_at_utc: 2026-05-10T22:12:11+00:00
calendar_strict: {
  "past_months": 3,
  "future_months": 6,
  "description": "rows require cube rows t-3..t+6 (lags at t-1,t-2,t-3)"
}
split_defaults: {
  "K_holdout_tail": 2,
  "J_val_before_holdout": 2,
  "note": "per-table tail ranks on anchor_month"
}
sample_weight: {
  "formula": "min(sqrt(n_articles_at_anchor), cap)",
  "cap": 100.0
}
univariate features: ['month_of_year', 'share_t', 'share_lag1', 'share_lag2', 'share_lag3']
fingerprint features: ['month_of_year', 'share_t', 'share_lag1', 'share_lag2', 'share_lag3']

univariate: rows_written=2,456 cols=18 stats={'n_groups': 191, 'n_candidates': 4325, 'n_rows': 2456}

fingerprint: rows_written=54,605 cols=21 stats={'n_groups': 18589, 'n_candidates': 196301, 'n_rows': 54605}


In [ ]:
def load_training(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run: python -m pipelines.monthly.features")
    df = pd.read_parquet(path)
    df["anchor_month"] = pd.to_datetime(df["anchor_month"]).dt.as_unit("ns")
    return df


df_u = load_training(TRAINING_UNIVARIATE_PARQUET)
df_f = load_training(TRAINING_FINGERPRINT_PARQUET)

print("univariate", df_u.shape)
print("fingerprint", df_f.shape)
display(df_u.head(3))
display(df_f.head(3))

## Schema, splits, anchors

`split_group` is assigned from tail ranks of distinct anchor months (`train` / `val` / `holdout`; defaults K=2, J=2 in `features.py`).

In [ ]:
for label, df in (("univariate", df_u), ("fingerprint", df_f)):
    print(f"\n=== {label} dtypes ===")
    display(df.dtypes.to_frame("dtype"))
    vc = df["split_group"].value_counts()
    vc = vc.reindex(["train", "val", "holdout"]).fillna(0).astype(int)
    print("split_group counts:\n", vc.to_string())
    am = df["anchor_month"]
    print(
        "anchor_month min/max:", am.min(), am.max(),
        "| distinct:", am.nunique(),
    )
    by_split = df.groupby("split_group", observed=True)["anchor_month"].agg(["min", "max", "nunique"])
    display(by_split)

## Weights and article counts

`sample_weight = min(sqrt(n_articles_at_anchor), cap)` with `cap` from `training_run.json`.

In [ ]:
for label, df in (("univariate", df_u), ("fingerprint", df_f)):
    print(f"\n=== {label} n_articles & sample_weight ===")
    desc = df[["n_articles", "sample_weight"]].describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    ).T
    display(desc)
    at_cap = (df["sample_weight"] >= contract["sample_weight"]["cap"] - 1e-9).mean()
    print(f"fraction of rows at weight cap (~= sqrt(n) capped): {at_cap:.2%}")


## Features and targets

Models use the same numeric feature block for both tables (`month_of_year` + share trajectory). Targets are `y_h1`…`y_h6` = `share_articles` at \(t+1\)…\(t+6\).

In [ ]:
def summarize_xy(df: pd.DataFrame, label: str) -> None:
    cols = [*FEATURE_CORE, *TARGET_COLS]
    missing = [c for c in cols if c not in df.columns]
    assert not missing, missing
    train = df[df["split_group"] == "train"]
    print(f"\n=== {label} (train rows={len(train):,}) feature/target describe ===")
    display(train[cols].describe().T)
    yt = train[TARGET_COLS].to_numpy()
    deltas = yt - train["share_t"].to_numpy()[:, None]
    print("train: mean(abs(y_h1 - share_t)):", float(np.mean(np.abs(deltas[:, 0]))))
    print("train: mean horizon-6 delta vs share_t:", float(np.mean(np.abs(deltas[:, -1]))))


summarize_xy(df_u, "univariate")
summarize_xy(df_f, "fingerprint")


In [ ]:
corr_u = df_u[FEATURE_CORE + TARGET_COLS].corr(numeric_only=True)
corr_f = df_f[FEATURE_CORE + TARGET_COLS].corr(numeric_only=True)
print("univariate corr(feature, y_h1):")
display(corr_u.loc[FEATURE_CORE, TARGET_COLS[0]].to_frame("corr_y_h1"))
print("fingerprint corr(feature, y_h1):")
display(corr_f.loc[FEATURE_CORE, TARGET_COLS[0]].to_frame("corr_y_h1"))


## Univariate: dimensions and cardinality

In [ ]:
print("distinct (dimension, level_id):", df_u.set_index(["dimension", "level_id"]).index.nunique())
dim_summary = (
    df_u.groupby("dimension", observed=False)
    .agg(rows=("anchor_month", "size"), n_levels=("level_id", "nunique"))
    .sort_values("rows", ascending=False)
)
display(dim_summary)


## Fingerprint: ID columns and entropy

In [ ]:
fp_ids = [
    "product_type_id",
    "gender_id",
    "color_master_id",
    "graphical_appearance_id",
    "material_id",
]
uniq_combo = df_f[fp_ids].drop_duplicates().shape[0]
print("distinct 5-D fingerprints in training rows:", uniq_combo)
for c in fp_ids:
    print(f"  {c}: nunique={df_f[c].nunique():,}")
display(df_f[fp_ids].describe(include="all").T.head(16))


## Optional quick plots (needs matplotlib)

`share_t` vs next-month target `y_h1` on a train subsample.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib not installed — skip plots")

if plt is not None:
    rng = np.random.default_rng(0)
    u_train = df_u[df_u["split_group"] == "train"]
    f_train = df_f[df_f["split_group"] == "train"]
    nu = min(8000, len(u_train))
    nf = min(8000, len(f_train))
    u_plot = u_train.iloc[rng.choice(len(u_train), size=nu, replace=False)]
    f_plot = f_train.iloc[rng.choice(len(f_train), size=nf, replace=False)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    for ax, (label, df) in zip(
        axes,
        [("univariate (train sample)", u_plot),
         ("fingerprint (train sample)", f_plot)],
    ):
        x = df["share_t"].to_numpy()
        y = df["y_h1"].to_numpy()
        ax.hexbin(x, y, gridsize=45, cmap="viridis", mincnt=1)
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
        ax.set_title(label)
        ax.set_xlabel("share_t")
        ax.set_ylabel("y_h1")
    plt.show()


## Latest trained model manifest (optional)

After `python -m pipelines.monthly.train`, compare holdout metrics to this notebook's distributional view.

In [ ]:
p = Path(MODEL_TRAINING_RUN_JSON)
if p.is_file():
    with open(p) as f:
        mtr = json.load(f)
    for key in ("univariate", "fingerprint"):
        block = mtr.get(key, {})
        ho = block.get("holdout_wmae")
        base = block.get("holdout_baseline_wmae")
        print(f"{key}: holdout wmae={ho} persistence_baseline={base}")
else:
    print("No", p, "— run pipelines.monthly.train after features.")
